# Quick start

## Loading a model
Here a simple guide to start:
https://cobrapy.readthedocs.io/en/latest/getting_started.html

In [ ]:
!pip install cobra 
from cobra.io import read_sbml_model

In [6]:
model = read_sbml_model('data/iJO1366.xml.gz')

In [7]:
model

Name,iJO1366
Memory address,11a511010
Number of metabolites,1805
Number of reactions,2583
Number of genes,1367
Number of groups,36
Objective expression,1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1
Compartments,"cytosol, extracellular space, periplasm"


In [8]:
model.reactions[29]

Reaction identifier,EX_5dglcn_e
Name,5-Dehydro-D-gluconate exchange
Memory address,0x11dc88aa0
Stoichiometry,5dglcn_e --> 5-Dehydro-D-gluconate -->
GPR,
Lower bound,0.0
Upper bound,1000.0


In [9]:
pgi = model.reactions.get_by_id("PGI")
pgi

Reaction identifier,PGI
Name,Glucose-6-phosphate isomerase
Memory address,0x2abac57b0
Stoichiometry,g6p_c <=> f6p_c D-Glucose 6-phosphate <=> D-Fructose 6-phosphate
GPR,b4025
Lower bound,-1000.0
Upper bound,1000.0


In [10]:
model.metabolites.get_by_id("atp_c")

Metabolite identifier,atp_c
Name,ATP
Memory address,0x11a5a9090
Formula,C10H12N5O13P3
Compartment,c
In 359 reaction(s),"2AGPGAT141, UAMAGS, ADOCBLabcpp, DPCOAK, 2AGPGAT160, ANHMK, UAMAS, GALabcpp, 2AGPGAT161, ISETACabcpp, COLIPAPabctex, HEPK1, 2AGPGAT180, TDSK, COLIPAabcpp, HEPK2, 2AGPGAT181, K2L4Aabcpp, METAT,..."


In [11]:
model.reactions.EX_glc__D_e.bounds

(-10.0, 1000.0)

The GPR is stored as the GPR class in the gpr for a Reaction. A string representation of it is stored as the gene_reaction_rule for a Reaction object.

In [12]:
gpr = pgi.gpr
print(gpr)
gpr_string = pgi.gene_reaction_rule
print(gpr_string)

b4025
b4025


In [13]:
pgi.genes
pgi_gene = model.genes.get_by_id("b4025")
pgi_gene

Gene identifier,b4025
Name,pgi
Memory address,0x11de1f5d0
Functional,True
In 1 reaction(s),PGI


In [18]:
model.compartments

{'c': 'cytosol', 'e': 'extracellular space', 'p': 'periplasm'}

Let's take a closer look at the reactions associated with Glyceraldehyde 3-phosphate (`g3p`).

In [19]:
for reaction in model.metabolites.g3p_c.reactions:
    print(reaction, reaction.name)

TRPS3: 3ig3p_c --> g3p_c + indole_c Tryptophan synthase (indoleglycerol phosphate)
TKT2: e4p_c + xu5p__D_c <=> f6p_c + g3p_c Transketolase
TGBPA: tagdp__D_c <=> dhap_c + g3p_c Tagatose-bisphosphate aldolase
TALA: g3p_c + s7p_c <=> e4p_c + f6p_c Transaldolase
DDPGALA: 2dh3dgal6p_c <=> g3p_c + pyr_c 2-dehydro-3-deoxy-6-phosphogalactonate aldolase
TKT1: r5p_c + xu5p__D_c <=> g3p_c + s7p_c Transketolase
TRPS1: 3ig3p_c + ser__L_c --> g3p_c + h2o_c + trp__L_c Tryptophan synthase (indoleglycerol phosphate)
TPI: dhap_c <=> g3p_c Triose-phosphate isomerase
F6PA: f6p_c <=> dha_c + g3p_c Fructose 6-phosphate aldolase
DXPS: g3p_c + h_c + pyr_c --> co2_c + dxyl5p_c 1-deoxy-D-xylulose 5-phosphate synthase
FBA: fdp_c <=> dhap_c + g3p_c Fructose-bisphosphate aldolase
EDA: 2ddg6p_c --> g3p_c + pyr_c 2-dehydro-3-deoxy-phosphogluconate aldolase
DRPA: 2dr5p_c --> acald_c + g3p_c Deoxyribose-phosphate aldolase
GAPD: g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c Glyceraldehyde-3-phosphate dehydrogenase


In the iJO1366 model, the default objective is to maximize the flux through the biomass reaction (i.e. growth). 

In [20]:
print(model.objective)

Maximize
1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


## Simulations

In [14]:
model.solver

In [16]:
solution = model.optimize()

Check the growth rate:

In [17]:
model.reactions.query("BIOMASS")

[<Reaction BIOMASS_Ec_iJO1366_WT_53p95M at 0x11dc845a0>,
 <Reaction BIOMASS_Ec_iJO1366_core_53p95M at 0x11dc84380>]

In [8]:
solution.fluxes.BIOMASS_Ec_iJO1366_core_53p95M

0.98237181272698204

We can investigate all fluxes by creating a data frame and getting all fluxes that are greater than 0.0001 mmol/g* DWh$^{-1}$

In [9]:
solution_frame = solution.to_frame()
solution_frame[solution_frame.fluxes.abs() > 1e-4]

,fluxes,reduced_costs
DM_4crsol_c,0.0002,0.0000e+00
DM_5drib_c,0.0002,0.0000e+00
DM_mththf_c,0.0004,0.0000e+00
BIOMASS_Ec_iJO1366_core_53p95M,0.9824,1.8492e-15
EX_ca2_e,-0.0051,0.0000e+00
...,...,...
UPPDC1,0.0002,0.0000e+00
USHD,0.0191,0.0000e+00
VALTA,-0.4157,0.0000e+00
ZN2tpp,0.0003,0.0000e+00
